[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pradeepvaka/llm-inference-90day/blob/master/notebooks/day39-kv-eviction-h2o-snapkv-streamingllm.ipynb)

# Day 39 — KV Eviction: H2O, SnapKV, StreamingLLM

**Big idea:** the KV cache gets an eviction policy. Attention mass concentrates on attention sinks (the first few tokens) and heavy hitters, plus the recent window — so a fixed budget can replace keep-everything. Today is CPU-only synthetic math, no model download: you will (1) price the full cache vs StreamingLLM at 100k context, (2) implement the sink + sliding-window cache and prove it stays bounded, (3) reproduce the evict-token-0 sink ablation on a toy attention row, (4) select H2O heavy hitters from synthetic cumulative scores, (5) run a simplified per-head SnapKV vote, and (6) fail a needle-in-the-middle test on purpose. ~30-60 min.


In [ ]:
# Cell 1: one-time installs (only cell that installs)
%pip install --quiet numpy matplotlib
print("deps ok")
# Expected: deps ok


## Step 1 — Memory math: full cache vs StreamingLLM

Llama-3.1-8B geometry (Day 8): 32 layers, 8 KV heads (GQA), head_dim 128, fp16.
Bytes/token = 2 (K,V) x layers x kv_heads x head_dim x dtype_bytes.


In [ ]:
import numpy as np

def kv_bytes_per_token(layers=32, kv_heads=8, head_dim=128, dtype_bytes=2):
    return 2 * layers * kv_heads * head_dim * dtype_bytes

per_token = kv_bytes_per_token()
KB = per_token / 1024  # 128 KB per token; headline GB/MB below use the packet's 128 KB shorthand
print(f"bytes/token = {per_token} = {KB:.0f} KB")
for seq in (4096, 32768, 100000):
    gb = seq * KB / 1e6
    print(f"full cache @ {seq:>6} tokens: {gb:6.2f} GB")
kept = 4 + 2040  # StreamingLLM: 4 sinks + 2040 recent window
stream_mb = kept * KB / 1e3
print(f"StreamingLLM kept entries = {kept}: {stream_mb:.0f} MB")
print(f"compression at 100k context = {100000/kept:.1f}x smaller")
assert per_token == 131072
assert abs(100000 * KB / 1e6 - 12.8) < 0.01
assert kept == 2044
# Expected: 128 KB/token; full cache ~0.52 / 4.19 / 12.80 GB
# Expected: StreamingLLM ~262 MB, ~48.9x smaller


## Step 2 — StreamingKVCache: sinks + sliding window

Policy: keep the first `sink` token ids forever, keep the last `window` ids, drop the middle. Order is preserved. Size must never exceed sink + window.


In [ ]:
class StreamingKVCache:
    def __init__(self, sink=4, window=2040):
        self.sink = sink
        self.window = window
        self.ids = []  # stands in for the cached token positions

    def append(self, token_id):
        self.ids.append(token_id)
        cap = self.sink + self.window
        if len(self.ids) > cap:
            # drop the middle: first `sink` + last `window`
            self.ids = self.ids[:self.sink] + self.ids[-self.window:]

    def __contains__(self, token_id):
        return token_id in self.ids

    def __len__(self):
        return len(self.ids)

cache = StreamingKVCache(sink=4, window=2040)
for t in range(10000):
    cache.append(t)
print(f"appended 10000 tokens, retained = {len(cache)}")
print(f"first retained ids = {cache.ids[:6]}")
print(f"last retained ids  = {cache.ids[-3:]}")
print(f"token 5000 retained? {5000 in cache}   token 9999 retained? {9999 in cache}")
assert len(cache) == 2044
assert cache.ids[:4] == [0, 1, 2, 3]
assert cache.ids[-1] == 9999
assert 5000 not in cache and 9999 in cache
print("verified: bounded at sink + window, sinks kept, middle evicted")
# Expected: retained = 2044; first ids 0,1,2,3; last id 9999; token 5000 absent


## Step 3 — The sink ablation: evict token 0 and watch the row swing

Toy attention row [0.62, 0.04, 0.10, 0.09, 0.15]: token 0 (the sink) holds more mass than the other four combined. Compute the attention output with distinct toy value vectors, then evict token 0, renormalize the survivors, and measure the output shift.


In [ ]:
w = np.array([0.62, 0.04, 0.10, 0.09, 0.15])
V = np.array([[1.0, 0.0],
              [0.0, 1.0],
              [1.0, 1.0],
              [-1.0, 1.0],
              [0.0, -1.0]])
out_full = w @ V
w_evict = w[1:] / w[1:].sum()
out_evict = w_evict @ V[1:]
print("renormalized row after evicting token 0:", np.round(w_evict, 3).tolist())
print(f"output with sink    = {np.round(out_full, 3).tolist()}")
print(f"output without sink = {np.round(out_evict, 3).tolist()}")
shift = np.linalg.norm(out_evict - out_full)
rel = shift / np.linalg.norm(out_full)
print(f"output shift = {shift:.3f}  (relative shift {rel:.2f} of the output norm)")
assert np.allclose(w_evict, [0.105, 0.263, 0.237, 0.395], atol=0.001)
assert rel > 0.5
print("verified: removing the 'empty' sink token swings the output by >50% of its norm")
# Expected: renormalized row ~ [0.105, 0.263, 0.237, 0.395]
# Expected: a large relative shift -- sinks are ballast the softmax depends on


## Step 4 — H2O: keep the heavy hitters by cumulative attention

Synthetic cumulative-attention scores with a heavy tail (Zipf-shaped, shuffled): a minority of tokens holds most of the mass. Policy: budget 1024 = top 512 by cumulative score + last 512 (recent window).


In [ ]:
rng = np.random.default_rng(42)
n = 8192
ranks = np.arange(1, n + 1)
scores = 1.0 / ranks ** 1.2          # heavy tail: rank 1 >> rank 8192
rng.shuffle(scores)                  # heavy hitters are scattered through the sequence
cum = scores / scores.sum()          # normalize to a mass distribution

BUDGET_HEAVY, WINDOW = 512, 512
heavy_idx = set(np.argsort(cum)[-BUDGET_HEAVY:].tolist())
window_idx = set(range(n - WINDOW, n))
kept_idx = heavy_idx | window_idx
mass_kept = cum[sorted(kept_idx)].sum()
print(f"kept {len(kept_idx)} of {n} entries ({n/len(kept_idx):.0f}x compression)")
print(f"fraction of total attention mass kept = {mass_kept:.3f}")
print(f"overlap heavy/window = {len(heavy_idx & window_idx)} tokens (counted once)")
assert len(kept_idx) <= BUDGET_HEAVY + WINDOW
assert mass_kept > 0.5
print("verified: a minority of entries holds a clear majority of the attention mass")
# Expected: <=1024 kept of 8192 (8x); mass kept is a clear majority (>0.5)


## Step 5 — SnapKV (simplified): a per-head vote from the observation window

Each head gets an attention matrix from a 32-token observation window (the prompt's tail) over a 512-token prompt. Different heads prefer different prompt regions (seeded synthetic preferences). Each head keeps its own top 128 positions.


In [ ]:
rng = np.random.default_rng(7)
PROMPT, HEADS, OBS, KEEP = 512, 4, 32, 128
kept_per_head = []
for h in range(HEADS):
    attn = rng.random((OBS, PROMPT)) * 0.1               # background noise
    pref_start = h * 100                                  # head h's preferred region
    attn[:, pref_start:pref_start + 100] += 1.0           # strong preference
    votes = attn.sum(axis=0)                              # observation window votes per position
    kept = set(np.argsort(votes)[-KEEP:].tolist())
    kept_per_head.append(kept)
    print(f"head {h}: keeps {len(kept)} positions, preferred region starts at {pref_start}")
overlap01 = len(kept_per_head[0] & kept_per_head[1])
print(f"overlap between head 0 and head 1 kept sets = {overlap01} of {KEEP}")
for s in kept_per_head:
    assert len(s) == KEEP
assert overlap01 < 80
print("verified: per-head kept sets differ materially -- that is the point of deciding per head")
# Expected: each head keeps exactly 128 positions
# Expected: head0/head1 overlap well below 128 (assert < 80)


## Step 6 — The needle test: the failure mode, on purpose

Reuse the Step 2 cache shape at small scale so the positions are readable: sink 4 + window 2040, 10,000 tokens appended, one needle at position 5000 (evicted middle) and one at 9990 (recent window). Then write the three-method contrast from memory and check it against the printed table.


In [ ]:
cache2 = StreamingKVCache(sink=4, window=2040)
for t in range(10000):
    cache2.append(t)
needle_mid, needle_recent = 5000, 9990
print(f"needle in the evicted middle (pos {needle_mid}): present = {needle_mid in cache2}")
print(f"needle in the recent window  (pos {needle_recent}): present = {needle_recent in cache2}")
assert needle_mid not in cache2 and needle_recent in cache2
print()
print("Method contrast:")
print("  StreamingLLM: position only, no scoring, bounded at sink + window forever")
print("  H2O:          cumulative attention during decode + recent window, fixed budget")
print("  SnapKV:       per-head vote by the prompt-end observation window, decided at prefill")
print()
print("Task split: local coherence (story, chat, code) survives eviction;")
print("mid-context recall (needle-in-a-haystack) breaks -- by construction, not bad luck.")
# Expected: middle needle absent (FAIL), recent needle present (PASS)


## Wrap-up

You should now be able to say, with numbers: full KV for 8B at 100k context is ~12.8 GB in the packet's rounding; StreamingLLM (4 + 2040) holds it to ~262 MB (~49x); sinks are softmax ballast, not content; H2O trusts accumulated use, SnapKV trusts a prompt-end per-head vote, StreamingLLM trusts position alone; and every policy fails the same workload — a fact stated once in the middle and needed at the end.

Tomorrow (Day 40): Mistral sliding window attention — the window trained into the architecture, a rolling buffer cache, and a W x layers effective receptive field.
